In [ ]:
# Essay #10: Adaptive Attacks Against Alternative Uncertainty
# Complete Notebook
# -----------------------------------------------------------
# Imports
# Part A: Dataset
# Part B: Models
# Part C: Attacks
# Part D: Calibration & Evaluation
# Part E: Main Execution
#------------------------------------------------------------


In [ ]:
# ─────────────────────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import json

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part A: Dataset
# ─────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

calib_size = 2000
eval_size = len(test_dataset) - calib_size
calib_dataset, eval_dataset = random_split(test_dataset, [calib_size, eval_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part B: Models
# ─────────────────────────────────────────────────────────────

class MCDropoutCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.3)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

class EvidentialCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten()
        )
        self.fc = nn.Linear(64 * 8 * 8, 10)
    def forward(self, x):
        return F.softplus(self.fc(self.features(x))) + 1.0

# Helper to train models
def train_model(model, is_evidential=False):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(15): # 15 epochs for speed
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            
            if is_evidential:
                alpha = model(images)
                S = alpha.sum(dim=1, keepdim=True)
                loss = F.nll_loss(torch.log(alpha / S + 1e-10), labels)
            else:
                logits = model(images)
                loss = F.cross_entropy(logits, labels)
                
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} completed.")
    return model


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part C: Attacks
# ─────────────────────────────────────────────────────────────

def pgd_standard(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    adv = images.clone().detach()
    for _ in range(num_steps):
        adv.requires_grad = True
        logits = model(adv)
        loss = F.cross_entropy(logits, labels)
        model.zero_grad()
        loss.backward()
        with torch.no_grad():
            adv = adv + step_size * adv.grad.sign()
            perturbation = torch.clamp(adv - images, -epsilon, epsilon)
            adv = torch.clamp(images + perturbation, 0, 1).detach()
    return adv

def adaptive_attack_mc(model, images, labels, epsilon=0.03, step_size=0.007,
                       num_steps=40, lam=1.0, k_passes=10):
    model.train()  # dropout ON: attack the actual stochastic gate
    original = images.clone().detach()
    adv = original.clone()
    for _ in range(num_steps):
        adv.requires_grad_(True)
        probs = torch.stack([F.softmax(model(adv), dim=1) for _ in range(k_passes)])
        cls_loss = F.nll_loss(torch.log(probs.mean(0) + 1e-10), labels)
        var_term = probs.var(0).max(1).values.mean()   # the real gate signal
        loss = cls_loss - lam * var_term               # ascend: error up, variance down
        grad = torch.autograd.grad(loss, adv)[0]
        step = adv.detach() + step_size * grad.sign()
        adv = clamp_valid(original + torch.clamp(step - original, -epsilon, epsilon)).detach()

    return adv

def adaptive_attack_evid(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40, lam=5.0):
    model.eval()
    adv = images.clone().detach()
    for _ in range(num_steps):
        adv.requires_grad = True
        alpha = model(adv)
        S = alpha.sum(dim=1, keepdim=True)
        probs = alpha / S
        loss_cls = F.nll_loss(torch.log(probs + 1e-10), labels)
        # Suppress deferral by maximizing evidence
        loss_evid = S.squeeze(1).mean()
        loss = loss_cls - lam * loss_evid * 0.01
        model.zero_grad()
        loss.backward()
        with torch.no_grad():
            adv = adv + step_size * adv.grad.sign()
            perturbation = torch.clamp(adv - images, -epsilon, epsilon)
            adv = clamp_valid(images + torch.clamp(step - original, -epsilon, epsilon)).detach()
    return adv


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part D: Calibration & Evaluation
# ─────────────────────────────────────────────────────────────

def calibrate_mc(model, loader, target=0.25):
    model.train()
    variances = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE)
            logits = torch.stack([model(images) for _ in range(10)])
            var = F.softmax(logits, dim=2).var(dim=0).max(dim=1)[0]
            variances.extend(var.cpu().numpy())
    return np.percentile(variances, 100 * (1 - target))

def calibrate_evid(model, loader, target=0.25):
    model.eval()
    evidences = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE)
            alpha = model(images)
            evidences.extend(alpha.sum(dim=1).cpu().numpy())
    # Target 25% deferral means threshold at the 25th percentile (lowest evidence defer)
    return np.percentile(evidences, 100 * target)

def evaluate(model, loader, threshold, attack_type='clean', is_evidential=False):
    n_correct_pred = n_wrong_undeferred = n_pred = n_def = n_total = 0
    
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        if attack_type == 'transferred':
            images = pgd_standard(model, images, labels)
        elif attack_type == 'adaptive_mc':
            images = adaptive_attack_mc(model, images, labels)
        elif attack_type == 'adaptive_evid':
            images = adaptive_attack_evid(model, images, labels)
            
        if is_evidential:
            model.eval()
            with torch.no_grad():
                alpha = model(images)
                S = alpha.sum(dim=1, keepdim=True)
                probs = alpha / S
                evidence = S.squeeze(1)
            defer = evidence < threshold
        else:
            model.train()
            with torch.no_grad():
                logits = torch.stack([model(images) for _ in range(10)])
                probs = F.softmax(logits, dim=2).mean(dim=0)
                variance = F.softmax(logits, dim=2).var(dim=0).max(dim=1)[0]
            defer = variance > threshold
            
        preds = probs.argmax(dim=1)
        correct = (preds == labels)
        
        n_total += labels.size(0)
        n_def += int(defer.sum().item())
        n_pred += int((~defer).sum().item())
        n_correct_pred += int((correct & ~defer).sum().item())
        n_wrong_undeferred += int(((~correct) & (~defer)).sum().item())

    acc_on_pred = 100.0 * n_correct_pred / n_pred if n_pred else 0.0
    return {
        'risk_on_pred': 100.0 - acc_on_pred,
        'wrong_undeferred': 100.0 * n_wrong_undeferred / n_total,
        'deferral_rate': 100.0 * n_def / n_total
    }


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part E: Main Execution
# ─────────────────────────────────────────────────────────────

print("Training MC Dropout Model...")
mc_model = MCDropoutCNN().to(DEVICE)
mc_model = train_model(mc_model, is_evidential=False)

print("Training Evidential Model...")
evid_model = EvidentialCNN().to(DEVICE)
evid_model = train_model(evid_model, is_evidential=True)

print("Calibrating Thresholds...")
mc_thresh = calibrate_mc(mc_model, calib_loader)
evid_thresh = calibrate_evid(evid_model, calib_loader)
print(f"MC Threshold: {mc_thresh:.4f} | Evid Threshold: {evid_thresh:.4f}")

print("\nEvaluating MC Dropout...")
mc_clean = evaluate(mc_model, eval_loader, mc_thresh, 'clean', False)
mc_trans = evaluate(mc_model, eval_loader, mc_thresh, 'transferred', False)
mc_adapt = evaluate(mc_model, eval_loader, mc_thresh, 'adaptive_mc', False)

print("Evaluating Evidential...")
evid_clean = evaluate(evid_model, eval_loader, evid_thresh, 'clean', True)
evid_trans = evaluate(evid_model, eval_loader, evid_thresh, 'transferred', True)
evid_adapt = evaluate(evid_model, eval_loader, evid_thresh, 'adaptive_evid', True)

print("\n--- FINAL RESULTS ---")
print(f"MC Clean: {mc_clean}")
print(f"MC Transferred: {mc_trans}")
print(f"MC Adaptive: {mc_adapt}")
print(f"Evid Clean: {evid_clean}")
print(f"Evid Transferred: {evid_trans}")
print(f"Evid Adaptive: {evid_adapt}")
